# Soup Lab Light Profile

Diagnostic-only follow-up on qualified runtime77cfc4d, 28-layer offload and bulk H2D.
The earlier heavy trace inflated host launch gaps. Disable record_shapes, with_stack and profiler memory collection for this CPU/CUDA timeline.
Run separate natural startup phases and a three-step timeline in full 64-step children.
No memory-history serialization, runtime source change, cached compiler artifacts or accepted timing claims.
Source hashes and raw traces are exported in light-profile/stack-profile-raw.zip.


In [ ]:
import subprocess, sys
from pathlib import Path
TARGET = '77cfc4d7ef293b1d45e2dab06b5428d178f1faa1'
DRIVER_COMMIT = 'a0d9bff2a2d5803d81a71036e57b2651de96f134'
REPOSITORY = 'https://github.com/kok-o/soup-kernel-lab.git'
DRIVER = Path('/kaggle/working/light-profile-driver')
OUT = Path('/kaggle/working/light-profile')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
assert not DRIVER.exists() and not OUT.exists(), 'Use a fresh session'
run(['git', 'init', str(DRIVER)])
run(['git', '-C', str(DRIVER), 'config', 'core.autocrlf', 'false'])
run(['git', '-C', str(DRIVER), 'config', 'core.eol', 'lf'])
run(['git', '-C', str(DRIVER), 'remote', 'add', 'origin', REPOSITORY])
run(['git', '-C', str(DRIVER), 'fetch', '--depth', '1', 'origin', DRIVER_COMMIT])
run(['git', '-C', str(DRIVER), 'checkout', '--detach', 'FETCH_HEAD'])
assert subprocess.check_output(['git','-C',str(DRIVER),'rev-parse','HEAD'], text=True).strip() == DRIVER_COMMIT
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True))
run([sys.executable, 'research/test_phase_monitor.py'], cwd=DRIVER)


In [ ]:
# Installation is outside the official timed child processes.
run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao', 'xformers', 'flash-attn', 'liger-kernel', 'unsloth', 'unsloth-zoo'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'triton==3.6.0', 'soup-cli[train]==0.77.0', 'transformers==5.16.1', 'peft==0.20.0', 'trl==0.29.1', 'bitsandbytes==0.50.2', 'accelerate==1.14.0', 'datasets==4.8.5', 'pytest'])


In [ ]:
run([sys.executable, 'research/run_stack_profile.py', '--commit', TARGET, '--out', str(OUT), '--modes', 'phases', 'timeline', '--light-timeline'], cwd=DRIVER)


In [ ]:
import json
r=json.loads((OUT/'PROFILE_PROVENANCE.json').read_text())
print('DIAGNOSTIC ONLY:',r['commit'])
print(json.dumps({k:v['exit_code'] for k,v in r['results'].items()},indent=2))
